# APIM ❤️ Foundry IQ

## Foundry IQ Agent Service Integration via APIM lab

![flow](../../images/foundry-iq-agent-svc.gif)

This lab integrates the **Foundry Agent Service** with a **Foundry IQ Knowledge Base** through **MCP (Model Context Protocol)**, with **Azure API Management** serving as the gateway for all agent inference traffic. The agent uses the `knowledge_base_retrieve` MCP tool to answer questions with citations from an Azure AI Search Knowledge Base.

### Key concepts

- **Foundry IQ Knowledge Base** — Azure AI Search’s managed Agentic Retrieval pipeline
- **MCP Tool Integration** — Agent calls knowledge base via `knowledge_base_retrieve` MCP tool
- **APIM as AI Gateway** — All OpenAI inference traffic (embeddings + chat) flows through APIM with token metrics, rate limiting, and managed identity auth
- **Conversations API** — Multi-turn conversation invocation pattern for the Foundry Agent Service

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management)
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

# AI Services configuration
aiservices_config = [{"name": "foundry", "location": "swedencentral"}]

embedding_model = "text-embedding-3-large"
agent_model = "gpt-4.1-mini"

# Models configuration - GPT-4.1-mini for agent + text-embedding-3-large for vectorizer
models_config = [
    {"name": agent_model, "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50},
    {"name": embedding_model, "publisher": "OpenAI", "version": "1", "sku": "GlobalStandard", "capacity": 120}
]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# API configuration
inference_api_path = "inference"
inference_api_type = "AzureOpenAI"
foundry_project_name = deployment_name

# Knowledge Base configuration
index_name = "foundry-iq-idx"
knowledge_source_name = "foundry-iq-ks"
knowledge_base_name = "foundry-iq-kb"
project_connection_name = "foundry-iq-conn"

agent_name = "foundry-iq-agent"

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed in the specified resource group. This includes:

- **Azure API Management** — Gateway for all inference traffic with managed identity auth
- **Azure AI Foundry** — Cognitive Services account + Project for agent hosting
- **Azure AI Search** — Search service for Knowledge Base with semantic ranking
- **Log Analytics + Application Insights** — Monitoring and diagnostics

Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "searchServiceLocation": { "value": resource_group_location },
        "knowledgeBaseName": { "value": knowledge_base_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, subscription key, search endpoint, and Foundry project endpoint from the Bicep deployment.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key")
    foundry_project_endpoint = utils.get_deployment_output(output, 'foundryProjectEndpoint', 'Foundry Project Endpoint')
    search_endpoint = utils.get_deployment_output(output, 'searchServiceEndpoint', 'Search Service Endpoint')
    search_service_name = utils.get_deployment_output(output, 'searchServiceName', 'Search Service Name')
    ai_services_endpoint = utils.get_deployment_output(output, 'aiServicesEndpoint', 'AI Services Endpoint')
    ai_services_name = utils.get_deployment_output(output, 'aiServicesName', 'AI Services Name')
    kb_mcp_endpoint = utils.get_deployment_output(output, 'kbMcpEndpoint', 'KB MCP Endpoint')

<a id='4'></a>
### 4️⃣ Create Search Index with Agentic Retrieval Support

Create an Azure AI Search index configured for Foundry IQ Agentic Retrieval with:
- **Vector search** — HNSW algorithm with Azure OpenAI vectorizer
- **Semantic search** — Semantic reranking for improved relevance
- **Auto-vectorization** — Vectorizer calls OpenAI embeddings at query time

In [ ]:
%pip install 'azure-search-documents==11.7.0b2' 'azure-ai-projects>=2.0.0' 'azure-identity' 'openai'

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, VectorSearch, VectorSearchProfile,
    HnswAlgorithmConfiguration, AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField
)

credential = DefaultAzureCredential()
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

# Create search index with vector + semantic search
index = SearchIndex(
    name=index_name,
    fields=[
        SearchField(name="id", type="Edm.String", key=True, filterable=True, sortable=True),
        SearchField(name="title", type="Edm.String", searchable=True, filterable=True),
        SearchField(name="content", type="Edm.String", searchable=True),
        SearchField(name="content_vector", type="Collection(Edm.Single)", stored=False,
                    vector_search_dimensions=3072, vector_search_profile_name="vector_profile"),
        SearchField(name="source", type="Edm.String", filterable=True),
        SearchField(name="page_number", type="Edm.Int32", filterable=True, sortable=True),
        SearchField(name="category", type="Edm.String", filterable=True, facetable=True)
    ],
    vector_search=VectorSearch(
        profiles=[VectorSearchProfile(name="vector_profile",
                                      algorithm_configuration_name="hnsw_config",
                                      vectorizer_name="azure_openai_vectorizer")],
        algorithms=[HnswAlgorithmConfiguration(name="hnsw_config")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_vectorizer",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=ai_services_endpoint,   #### Direct AI Services endpoint to the vectorizer parameters
                    deployment_name=embedding_model,
                    model_name=embedding_model
                )
            )
        ]
    ),
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="content")]
                )
            )
        ]
    )
)

index_client.create_or_update_index(index)
utils.print_ok(f"Index '{index_name}' created successfully")

<a id='5'></a>
### 5️⃣ Upload sample documents

Generate embeddings via APIM gateway and upload documents to the search index. Note that embedding generation flows through the **APIM gateway**, demonstrating APIM-mediated traffic for AI operations.

In [ ]:
from azure.search.documents import SearchClient
from openai import AzureOpenAI

# Initialize OpenAI client pointing at APIM gateway
openai_client = AzureOpenAI(
    azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key=api_key,
    api_version="2024-06-01"
)

# Sample documents about Azure AI services
sample_documents = [
    {
        "id": "1",
        "title": "Foundry Agent Service Overview",
        "content": "Microsoft Foundry Agent Service allows you to create AI Agents customized with custom instructions and advanced tools like code interpreter and custom functions. Agents can use MCP (Model Context Protocol) tools to connect to Knowledge Bases for knowledge retrieval. The service supports both the Conversations API for simple interactions and the Classic Agent API for complex multi-turn workflows.",
        "source": "foundry-docs",
        "page_number": 1,
        "category": "agents"
    },
    {
        "id": "2",
        "title": "MCP Tool Integration",
        "content": "MCP (Model Context Protocol) is the standard protocol for Agent to Knowledge Base communication. By creating a RemoteTool project connection, Agents can securely call the knowledge_base_retrieve tool to retrieve relevant content and generate responses with citations. The connection uses ProjectManagedIdentity for secure authentication without managing API keys.",
        "source": "foundry-docs",
        "page_number": 2,
        "category": "integration"
    },
    {
        "id": "3",
        "title": "Agent Creation Best Practices",
        "content": "Best practices for creating Agents: 1) Use clear instructions telling the Agent it must use the knowledge base to answer questions; 2) Require the Agent to provide citations in answers; 3) If the answer cannot be found, respond with 'I don't know' rather than guess; 4) Use EXTRACTIVE_DATA output mode with minimal reasoning effort for Agent integration.",
        "source": "foundry-docs",
        "page_number": 3,
        "category": "best-practices"
    },
    {
        "id": "4",
        "title": "Foundry IQ Knowledge Base Configuration",
        "content": "Foundry IQ is the brand name for Azure AI Search's Agentic Retrieval capability. Knowledge Base is the core of the Agentic Retrieval pipeline. It defines which knowledge sources to query and the default behavior of retrieval operations. Key properties include: output_mode (answer_synthesis or extracted_data), retrieval_reasoning_effort (minimal, low, medium), and knowledge_sources (one or more knowledge source references).",
        "source": "azure-docs",
        "page_number": 4,
        "category": "configuration"
    },
    {
        "id": "5",
        "title": "Retrieval Reasoning Effort Levels",
        "content": "Retrieval Reasoning Effort determines the level of LLM-related query processing. minimal - bypasses LLM query planning, reducing cost and latency; low (default) - balanced LLM processing approach; medium - maximizes LLM processing for improved relevance. For Foundry Agent Service integration, minimal is recommended for better performance and lower cost.",
        "source": "azure-docs",
        "page_number": 5,
        "category": "performance"
    },
    {
        "id": "6",
        "title": "APIM as AI Gateway for Foundry IQ",
        "content": "Azure API Management can serve as an AI Gateway mediating all traffic between clients, Azure OpenAI, and Azure AI Search. APIM provides token rate limiting, semantic caching, load balancing across model deployments, and unified observability. When combined with Foundry IQ, APIM adds managed identity authentication, token metrics emission, and centralized access control to the knowledge retrieval pipeline.",
        "source": "gateway-docs",
        "page_number": 6,
        "category": "gateway"
    }
]

# Generate embeddings via APIM gateway
utils.print_info("Generating vector embeddings via APIM gateway...")
for doc in sample_documents:
    response = openai_client.embeddings.create(input=doc["content"], model=embedding_model)
    doc["content_vector"] = response.data[0].embedding
    utils.print_info(f"  ✓ {doc['title']}")

# Upload documents to search index
search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
result = search_client.upload_documents(documents=sample_documents)
succeeded = sum(1 for r in result if r.succeeded)
utils.print_ok(f"Uploaded {succeeded}/{len(sample_documents)} documents to index '{index_name}'")

<a id='6'></a>
### 6️⃣ Create Knowledge Source and Knowledge Base

Create the Foundry IQ primitives:
- **Knowledge Source** — References the search index and defines which fields to include in citations
- **Knowledge Base** — Configures the agentic retrieval pipeline with `EXTRACTIVE_DATA` output mode and `minimal` reasoning effort (recommended for Agent Service integration)

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters, SearchIndexFieldReference,
    KnowledgeBase, KnowledgeSourceReference,
    KnowledgeRetrievalOutputMode, KnowledgeRetrievalMinimalReasoningEffort, KnowledgeBaseAzureOpenAIModel
)

# Create Knowledge Source
knowledge_source = SearchIndexKnowledgeSource(
    name=knowledge_source_name,
    description="Foundry Agent Service and AI Gateway documentation",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=index_name,
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="title"),
            SearchIndexFieldReference(name="source"),
            SearchIndexFieldReference(name="page_number"),
            SearchIndexFieldReference(name="category")
        ]
    )
)

index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
utils.print_ok(f"Knowledge Source '{knowledge_source_name}' created")

In [ ]:
# Configure Azure OpenAI connection parameters
### uses proxied inteferencing via APIM gateway, pointing the vectorizer to APIM instead of AI Services directly
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=apim_resource_gateway_url,
    deployment_name=agent_model,
    model_name=agent_model,
    api_key=subscription_key
)

# Create Knowledge Base with EXTRACTIVE_DATA mode (recommended for Agent integration)
knowledge_base = KnowledgeBase(
    name=knowledge_base_name,
    description="Knowledge base for Foundry Agent Service with APIM Gateway documentation",
    knowledge_sources=[
        KnowledgeSourceReference(name=knowledge_source_name)
    ],
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],

    # retrieval_instructions= "",
    # answer_instructions= ""
)

index_client.create_or_update_knowledge_base(knowledge_base=knowledge_base)
utils.print_ok(f"Knowledge Base '{knowledge_base_name}' created")

# Build MCP endpoint URL
direct_mcp_endpoint = f"{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version=2025-11-01-Preview"
utils.print_info(f"Direct MCP Endpoint: {direct_mcp_endpoint}")

<a id='7'></a>
### 7️⃣ Create Project Connections (Cog Search & MCP types)

Create a `RemoteTool` connection in the AI Foundry project pointing to KB/Agentic Search endpoint. It allows agents to use the designated KB.
This connection uses the project’s **managed identity** for authentication — no API keys to manage.

Here we're using REST APIs to create the connection in Foundry, alternatively you can use Bicep to perform the same task.

In [ ]:
import requests

mcp_endpoint = f"{kb_mcp_endpoint}?api-version=2025-11-01-Preview"

# Get the project resource ID for Management API calls
output = utils.run(
    f'az resource list -g {resource_group_name} --resource-type "microsoft.cognitiveservices/accounts/projects" -o json',
    "Retrieved AI Foundry projects", "Failed to list AI Foundry projects")

project_resource_id = None
if output.success and output.json_data:
    for resource in output.json_data:
        utils.print_info(f"Found project: {resource['name']}")
        project_resource_id = resource['id']

if not project_resource_id:
    utils.print_error("No AI Foundry project found")
else:
    # Get Bearer Token for Management API
    bearer_token_provider = get_bearer_token_provider(credential, "https://management.azure.com/.default")
    headers = {
        "Authorization": f"Bearer {bearer_token_provider()}",
        "Content-Type": "application/json"
    }

    # Request body for creating RemoteTool connection
    connection_payload = {
        "name": f"{knowledge_base_name}-mcp",
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "CustomKeys",
            "credentials": {
                "keys": {
                    "api_key": subscription_key
                }
            },
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "metadata": {
                "ApiType": "Azure"
            }
        }
    }

    # Send create connection request
    connection_url = f"https://management.azure.com{project_resource_id}/connections/{knowledge_base_name}-mcp?api-version=2025-10-01-preview"
    response = requests.put(connection_url, headers=headers, json=connection_payload)

    if response.status_code in [200, 201]:
        utils.print_ok(f"Project Connection '{knowledge_base_name}-mcp' created")
        utils.print_info(f"Target: {mcp_endpoint}")
    else:
        utils.print_error(f"Failed to create connection: {response.status_code} - {response.text}")

It's optional to create a `CognitiveSearch` connection in the AI Foundry project pointing to AI Search endpoint. 
It allows exploring current KBs in Foundry UI (not needed)
This connection uses the project’s **managed identity** for authentication — no API keys to manage.

This one has already been created as part of the Bicep template so no need to rerun it, but this is another way of creating it if needed

In [ ]:
import requests

# Get the project resource ID for Management API calls
output = utils.run(
    f'az resource list -g {resource_group_name} --resource-type "microsoft.cognitiveservices/accounts/projects" -o json',
    "Retrieved AI Foundry projects", "Failed to list AI Foundry projects")

project_resource_id = None
if output.success and output.json_data:
    for resource in output.json_data:
        utils.print_info(f"Found project: {resource['name']}")
        project_resource_id = resource['id']

if not project_resource_id:
    utils.print_error("No AI Foundry project found")
else:
    # Get Bearer Token for Management API
    bearer_token_provider = get_bearer_token_provider(credential, "https://management.azure.com/.default")
    headers = {
        "Authorization": f"Bearer {bearer_token_provider()}",
        "Content-Type": "application/json"
    }

    # Request body for creating RemoteTool connection
    connection_payload = {
        "name": "aiSearchConnectionREST",
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "AAD",
            "credentials": {},
            "category": "CognitiveSearch",
            "target": f"https://{search_service_name}.search.windows.net/",
            "isSharedToAll": True,
            "metadata": {
                "ApiType": "Azure",
                "type": "azure_ai_search"
            }
        }
    }

    # Send create connection request
    connection_url = f"https://management.azure.com{project_resource_id}/connections/aiSearchConnectionREST?api-version=2025-10-01-preview"
    response = requests.put(connection_url, headers=headers, json=connection_payload)

    if response.status_code in [200, 201]:
        utils.print_ok(f"Project Connection 'aiSearchConnectionREST' created")
        utils.print_info(f"Target: https://{search_service_name}.search.windows.net")
    else:
        utils.print_error(f"Failed to create connection: {response.status_code} - {response.text}")

<a id='8'></a>
### 8️⃣ Create Agent with Knowledge Base

Create a Foundry Agent configured with the `knowledge_base_retrieve` tools. The agent will:
- Use the knowledge base to answer all questions
- Provide citations using the KB tool
- Respond with "I don't know" when the answer is not in the knowledge base

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Initialize Project Client
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=credential
)

# Define Agent instructions
agent_instructions = """You are a helpful assistant that must use the knowledge base to answer all the questions from user.
You must never answer from your own knowledge under any circumstances.

Every answer must always provide annotations for using the MCP knowledge base tool and render them as: [message_idx:search_idx|source_name]

If you cannot find the answer in the provided knowledge base you must respond with "I don't know".
"""

mcp_kb_tool = MCPTool(
    server_label=f"{knowledge_base_name}-mcp",
    server_url=mcp_endpoint,
    allowed_tools=["knowledge_base_retrieve"],
    require_approval="never",
    project_connection_id=f"{knowledge_base_name}-mcp",
)

# Create Agent
agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=agent_model,
        instructions=agent_instructions,
        tools=[mcp_kb_tool]
    )
)

utils.print_ok(f"Agent '{agent.name}' created (version: {agent.version})")
utils.print_info(f"Model: {agent_model}")

<a id='conversations'></a>
### 🧪 Test with Conversations API

The agent automatically calls the Knowledge Base’s MCP tool to retrieve relevant content and generate an answer with citations.

The Conversations API supports multi-turn interactions by sending follow-up messages to the same conversation. This replaces the Classic Agent API (Threads/Messages/Runs) which was removed in `azure-ai-projects` v2.x.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [ ]:
# Get OpenAI client for Conversations API
conversations_client = project_client.get_openai_client()

# Create a conversation
conversation = conversations_client.conversations.create()
utils.print_ok(f"Conversation created: {conversation.id}")

def chat_with_agent(question: str):
    """Send a question to the Agent and get a response with citations."""
    utils.print_info(f"Question: {question}")
    print("=" * 60)

    response = conversations_client.responses.create(
        conversation=conversation.id,
        tool_choice="required",             #force the Agent to use the MCP tool for retrieval
        input=question,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
                "version": agent.version
            }
        },
        stream=False
    )

    utils.print_message(f"Answer:\n{response.output_text}")
    return response

In [ ]:
# Test: Ask about MCP integration
response = chat_with_agent("How does MCP tool integration work with Foundry Agents?")

In [ ]:
# Test: Ask about retrieval reasoning effort
response = chat_with_agent("What are the Retrieval Reasoning Effort levels and which one is recommended for Agent integration?")

In [ ]:
# Test: Ask about APIM as AI Gateway
response = chat_with_agent("What role does Azure API Management play in the Foundry IQ architecture?")

In [ ]:
# Multi-turn: Ask a follow-up question in the same conversation
response = chat_with_agent("What are the best practices for creating Foundry Agents with knowledge base integration?")


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.